### 2.1 理论计算题

**已知：**
- 输入图像：3×32×32（通道数×高×宽）
- 卷积核：16个，每个尺寸 3×5×5（通道数×高×宽）
- Padding = 2，Stride = 2

**1. 输出特征图尺寸**

输出高度（或宽度）计算公式：
\[
H_{out} = \left\lfloor \frac{H_{in} + 2 \times padding - kernel\_size}{stride} \right\rfloor + 1
\]
代入：
\[
H_{out} = \left\lfloor \frac{32 + 2\times 2 - 5}{2} \right\rfloor + 1 
= \left\lfloor \frac{31}{2} \right\rfloor + 1 
= 15 + 1 = 16
\]
输出通道数 = 卷积核个数 = 16  
因此输出尺寸为：**16 × 16 × 16**（通道数×高×宽）

**2. 单个输出通道的一个像素值的点乘次数**

一个卷积核的参数数量 = 输入通道数 × 核高 × 核宽 = \(3 \times 5 \times 5 = 75\)  
每计算一个输出像素，需要将卷积核与输入对应区域做逐元素相乘并求和，即 75 次乘法。  
**答案：75 次**

In [1]:
import numpy as np

def max_pool2d(x, kernel_size, stride=None, padding=0):
    """
    x: (batch, channels, height, width)
    kernel_size: int or tuple
    stride: int or tuple, default=kernel_size
    padding: int or tuple
    """
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if stride is None:
        sh = sw = kh
    elif isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    batch, c, h, w = x.shape
    h_out = (h + 2*ph - kh) // sh + 1
    w_out = (w + 2*pw - kw) // sw + 1

    # 填充
    x_pad = np.pad(x, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant', constant_values=-np.inf)
    out = np.zeros((batch, c, h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            region = x_pad[:, :, i*sh:i*sh+kh, j*sw:j*sw+kw]
            out[:, :, i, j] = np.max(region, axis=(2,3))
    return out

### 3.1 理论计算题

假设输入和输出特征图通道数均为 \(C\)，且卷积层不带偏置。

**1. 一个 \(5 \times 5\) 卷积层的参数量**

参数量 = 输出通道数 × 输入通道数 × 卷积核高 × 卷积核宽  
\[
Params = C \times C \times 5 \times 5 = 25C^2
\]

**2. 两个串联的 \(3 \times 3\) 卷积层的总参数量**

每个 \(3 \times 3\) 卷积层参数量 = \(C \times C \times 3 \times 3 = 9C^2\)  
两个串联的总参数量 = \(9C^2 + 9C^2 = 18C^2\)

**结论**：两个 \(3 \times 3\) 卷积层（\(18C^2\)）比一个 \(5 \times 5\) 卷积层（\(25C^2\)）参数更少，且具有相同的感受野。

In [2]:
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.block(x)

### 4.1 理论计算题

**已知**：一个 batch 中某个通道某位置的特征值：\(x = [2, 4, 6, 8]\)  
缩放参数 \(\gamma = 2\)，平移参数 \(\beta = 1\)，\(\epsilon = 0\)

**步骤**：
1. 计算均值 \(\mu\)：
\[
\mu = \frac{2+4+6+8}{4} = 5
\]

2. 计算方差 \(\sigma^2\)（使用总体方差，因为 BN 默认除以 \(m\)）：
\[
\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} 
= \frac{9+1+1+9}{4} = \frac{20}{4} = 5
\]

3. 归一化：
\[
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}}
\]

4. 缩放平移：
\[
y_i = \gamma \hat{x}_i + \beta = 2 \cdot \frac{x_i - 5}{\sqrt{5}} + 1
\]

**最终输出**：
\[
y_1 = 1 - \frac{6}{\sqrt{5}}, \quad
y_2 = 1 - \frac{2}{\sqrt{5}}, \quad
y_3 = 1 + \frac{2}{\sqrt{5}}, \quad
y_4 = 1 + \frac{6}{\sqrt{5}}
\]

（数值近似：\(y_1 \approx -1.683,\; y_2 \approx 0.106,\; y_3 \approx 1.894,\; y_4 \approx 3.683\)）

In [3]:
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity() if in_channels == out_channels else None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        identity = x
        if self.shortcut is not None:
            identity = self.shortcut(x)
        out += identity
        return self.relu(out)

### 5.1 理论计算题

**1. 为什么底层用小学习率（或冻结），顶层用大学习率？**

- 底层（靠近输入）学习到的特征是通用的（边缘、纹理、颜色等），这些特征在大多数视觉任务中都有用。使用小学习率可以避免破坏这些良好的特征。
- 顶层（靠近输出）负责特定类别的高层语义，需要随机初始化或重新训练，因此需要较大学习率来快速适应新任务的类别分布。

**2. 目标数据集非常小且与源数据集非常相似时的微调策略**

- 冻结所有或大部分底层卷积层，只重新训练最后的全连接层（分类层）。
- 使用非常小的学习率（例如源模型学习率的 1/10），以避免对预训练权重造成大的破坏。
- 使用更强的正则化：数据增广（随机裁剪、翻转等）、Dropout、权重衰减等。
- 如果需要微调更多层，建议逐层解冻，并继续使用小学习率。

In [6]:
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

AttributeError: type object 'torch._C.Tag' has no attribute 'needs_exact_strides'

### 6.1 理论计算题

**已知**：  
真实框 \(A = [10,10,50,50]\)（左上角 x, 左上角 y, 右下角 x, 右下角 y）  
预测框 \(B = [30,30,70,70]\)

**步骤**：
1. 计算交集矩形的左上角坐标：
\[
x_{min} = \max(10, 30) = 30, \quad y_{min} = \max(10, 30) = 30
\]

2. 计算交集矩形的右下角坐标：
\[
x_{max} = \min(50, 70) = 50, \quad y_{max} = \min(50, 70) = 50
\]

3. 交集面积（若 \(x_{max} > x_{min}\) 且 \(y_{max} > y_{min}\)）：
\[
\text{area\_intersection} = (50 - 30) \times (50 - 30) = 20 \times 20 = 400
\]

4. 各自面积：
\[
\text{area}_A = (50 - 10) \times (50 - 10) = 40 \times 40 = 1600
\]
\[
\text{area}_B = (70 - 30) \times (70 - 30) = 40 \times 40 = 1600
\]

5. 并集面积：
\[
\text{area\_union} = \text{area}_A + \text{area}_B - \text{area\_intersection} = 1600 + 1600 - 400 = 2800
\]

6. IoU：
\[
IoU = \frac{\text{area\_intersection}}{\text{area\_union}} = \frac{400}{2800} = \frac{1}{7}
\]

**答案**：\(\frac{1}{7}\) （约 0.142857）

In [5]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(pred, target, epsilon=0.1):
    """pred: (batch, K) logits, target: (batch) class indices"""
    K = pred.size(-1)
    log_probs = F.log_softmax(pred, dim=-1)
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs)
        true_dist.fill_(epsilon / (K - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1 - epsilon)
    return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))